# GEMS G01 — physics-informed wavelet fault discovery

**Question (falsifiable).** Does label-free fusion of magnetic and gravity wavelet evidence — spatially tolerant co-location, axial strike compatibility and multiscale persistence — place more of an equal-area prediction budget near *held-out* faults than either field alone, than a simple gradient edge, and than M9's engineered cross-modal layers?

| ID | map | hypothesis | contrast tested |
|---|---|---|---|
| G01-A | magnetic wavelet energy (multiscale percentile) | magnetic structure is independently informative | A vs edge, A vs random |
| G01-B | gravity wavelet energy | density boundaries carry complementary evidence | B vs edge, B vs random |
| G01-C | (M + G + J) / 3, J = tolerant co-location | joint evidence beats either field alone | C vs A, C vs B |
| G01-D | C with J gated by axial strike agreement | strike alignment separates coherent structures | D vs C |
| G01-E | D x (floor + (1 - floor) persistence) | multiscale continuity helps | E vs D |
| G01-F | percentiles of M9 `m9:xmod:mag_grav_tol / orient` | established engineered features | E vs F, C vs F, F vs edge |

No candidate has a learned weight. Labels are used **only** to evaluate. G01-E is not assumed to win.

**Rules of this notebook**
* Read-only on M9: the cached S02 / S03 layers in `cache/m9/layers` are reused by name; no M9 section (S01–S19) is re-run, no network is built, no checkpoint is touched.
* G01 writes only under `outputs/m9_geofault_production/research/G01_wavelet_geology/g01_<fingerprint>/` (write-once; another configuration or input = another folder).
* **Exploratory** maps (sections 2, figures 1–8) are label-free and use every valid pixel as the percentile reference: they are not evidence of discovery. **Validated** numbers (sections 3–4, figures 9–12) come from held-out zones with percentile references fitted on each zone's training geography.
* This evaluation is separate from the official competition score (an official-style DTI of the same masks is shown for orientation only). The zones were inspected in earlier GEMS work: not an untouched test set.
* **No number exists until the cells run.**

Code: `gems_bench/m9/research/` · configuration: `configs/m9/g01_wavelet_geology.yaml` · guide: `docs/m9/G01_GEOLOGICAL_WAVELET.md`.

## 0. Configuration, Drive and context

Same workflow as the M9 notebook: mount Drive, install the repository at `CONFIG['ref']`, open the pipeline context (data stages validated / staged from Drive, never regenerated) and create the `m9` object **without running any M9 section**. G01 needs from M9 only its context (grid, validity mask, folds, discovery splits, units, eligible budget) and its layer cache.

Prerequisite: M9 section 03 (`s03_spectral`) and, for G01-F, section 02 (`s02_features`) have run once, so `cache/m9/layers` holds the log-Gabor and cross-modal layers. If the cache is not on Drive, set `G01_CONFIG['build_missing'] = {'wavelets': True, 'crossmodal': True}`: only the missing layers are computed with M9's own functions (and pushed to Drive for later sessions).

In [ ]:
CONFIG = {                                         # master-pipeline context (data stages are only validated / staged)
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'gems3-discovery-research',               # branch or commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False, 'train_baselines': [], 'run_full_inference': False, 'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': False,                                # G01 is CPU only (FFT-free: it reads cached wavelet layers)
    'seed': 2026,
}
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
PRIVATE_REPO = False

M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}   # only opens the M9 context; no section is run

G01_CONFIG = {
    'overwrite': False,                              # True rewrites files inside the SAME fingerprinted run folder
    'wavelengths_km': 'auto',                        # every wavelength cached for both fields
    'normalization': {'reference': 'training_geography'},
    'tolerance': {'min_m': 200.0, 'per_wavelength': 0.25, 'max_m': 1200.0, 'footprint': 'square'},
    'persistence': {'q': 0.8, 'floor': 0.5},
    'boundary_margin_m': 400.0,
    'evaluation': {'tasks': ['geo', 'disc_official'], 'zones': 'all', 'n_boot': 500},
    'exploratory': {'full_region': True},
    'build_missing': {'wavelets': False, 'crossmodal': False},
    'save': {'push_to_drive': True, 'geotiff': False},
}
SHOW_FIGURES = True

In [ ]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed:\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('code @', HEAD)
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('Code changed after gems_bench was imported: Runtime -> Restart session, then run from the top.')

In [ ]:
import json, pathlib
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.pipeline import Pipeline
from gems_bench.m9.research import G01Experiment

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for s in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(s)                                     # validated on Drive and staged locally (never regenerated)

m9 = pipe.m9(M9_CONFIG)                              # context only: no M9 section, no model, no training
g01 = G01Experiment(m9, G01_CONFIG)

def show(names):
    if not SHOW_FIGURES:
        return
    for n in names:
        p = g01.store.root / 'figures' / f'{n}.png'
        display(Image(filename=str(p))) if p.exists() else print('not produced:', n)

def tab(name, n=40):
    t = g01.store.load_table(name, 'tables')
    display(t.head(n)) if len(t) else print('empty / not produced:', name)
    return t

## 1. Inputs: cache audit, wavelength alignment, tolerances

* **What.** List the cached M9 layers, report every (field, wavelength, product) as present / missing / duplicate / unaligned, keep the wavelengths cached for **both** fields, fingerprint the inputs and open the versioned run folder.
* **Physics.** Each scale gets a lateral tolerance `tol = clip(0.25 λ, 200 m, 1200 m)` in metres (converted with the grid's actual pixel size): a quarter wavelength is the positional uncertainty of an edge seen at λ, and gravity sources are deeper and the survey coarser than the magnetic one.
* **Read.** `tables/channel_audit`, `tables/wavelength_tolerances`, warnings (grid not 100 m, G01-F unavailable).
* **Resume.** Always cheap; re-running it with the same inputs re-opens the same run folder.

In [ ]:
info = g01.prepare()
print('run:', info['run_id'], '->', g01.store.root)
print('wavelengths:', [w.label for w in info['wavelengths']])
print('grid:', info['grid'])
for w in info['warnings']:
    print('WARNING', w)
display(info['audit'].groupby(['role', 'status']).size().rename('n').reset_index())
tab('wavelength_tolerances')

## 2. Exploratory label-free evidence maps (full region)

* **What.** All six candidates and every component (M, G, J, J_D, persistence, orientation agreement / confidence / gate, surface expression S) over the whole region, tiled with halos so tolerant filters see real neighbours.
* **Status: EXPLORATORY.** The percentile reference is every valid pixel. These maps describe structural evidence; they are **not** verified fault discovery. High wavelet energy also marks lithological contacts, intrusion margins and dykes.
* **Artifacts.** `rasters/exploratory_full_region.npz` (float16, with an `interior` mask), optional GeoTIFFs.
* **Resume.** Reused when `processed/done_exploratory.json` exists. Skip with `G01_CONFIG['exploratory']['full_region'] = False`.

In [ ]:
expl = g01.exploratory()
if expl is not None:
    print({k: (v.shape, str(v.dtype)) for k, v in list(expl.items())[:6]}, '...')
    del expl

## 3. Held-out evaluation (geographic folds and discovery splits)

* **Units.** Every geographic fold (`geo`: truth = all mapped faults of the held-out fold) and every discovery split (`disc_official`: truth = withheld systems, known faults masked pixel-exactly) — `evaluation.zones = 'all'`, independent of the M9 training profile.
* **Leakage guards.** Percentile tables are fitted on the unit's training geography only (`assert_disjoint` raises on any overlap with the evaluation zone); scores never see labels; budgets are area fractions; nothing is tuned.
* **Domain.** M9's eligible budget domain ∩ pixels ≥ 400 m from the data edge — identical for every candidate, the simple edge and the random baseline. Exactly `k = round(frac · |domain|)` pixels per map.
* **Artifacts.** `processed/blocks_<task>_<zone>` (counts per 20 km block, candidate, budget, distance), `processed/official_<unit>`, `processed/refs_<unit>.json` (percentile tables), `rasters/units/<unit>.npz` (all maps + domain; labels are never stored).
* **Resume.** Per unit (`processed/done_unit_*.json`): an interrupted session continues with the next unit.

In [ ]:
units = g01.evaluate()
display(units)

## 4. Metrics, uncertainty and hypothesis verdicts

* **Metrics** at each distance d (0–1000 m, Euclidean on the real pixel size): precision@d, recall@d, F1@d, enrichment@d = precision@d / (share of the domain within d of truth) — random placement gives 1 — and the precision ratio vs the simple edge at the same budget.
* **Uncertainty.** Paired spatial-block bootstrap (20 km blocks resampled within each zone; same draw for both maps). Pixels are never treated as independent.
* **Verdicts** (`tables/hypothesis_contrasts`): SUPPORTED when Δ > 0, P(Δ > 0) ≥ 0.9 and the candidate wins on ≥ 67 % of zones; REFUTED when Δ < 0 and P(Δ > 0) ≤ 0.1; otherwise INCONCLUSIVE. Thresholds live in `decision` (configuration), fixed before the run.
* **Ablations.** Per wavelength (A/B/C/D@λ) and per fixed tolerance (C|tol=…); any 'best' variant is chosen leave-one-zone-out (`tables/nested_choice`), never on the zone it is scored on.
* **Official-style DTI** of the same masks: `tables/official_style_pooled` — orientation only, not the G01 criterion.

In [ ]:
res = g01.metrics()
cols = ['hypothesis', 'task', 'candidate', 'reference', 'metric', 'delta', 'ci_lo', 'ci_hi', 'p_delta_positive',
        'zone_win_share', 'verdict']
display(res['contrasts'][[c for c in cols if c in res['contrasts']]])
f0, d0 = g01.cfg['evaluation']['primary_area_frac'], g01.cfg['evaluation']['primary_distance_m']
P = res['pooled']
display(P[(P.area_frac == f0) & (P.tol_m == d0) & P.candidate.isin(g01.main_names())]
        [['task', 'candidate', 'precision', 'recall', 'f1', 'enrichment', 'precision_vs_edge']])
display(res['nested'])
display(res['official_pooled'])

## 5. Figures

One geographic window (50 km) for maps 1–8, identical extent and colour scale across panels of the same quantity; UTM 11N kilometres.

1 raw fields · 2 magnetic energy by scale · 3 gravity energy by scale · 4 strike + coherence · 5 orientation agreement · 6 single vs joint · 7 persistence · 8 six-way comparison (**exploratory**) · 9 equal-area selections vs held-out labels (**evaluation only**) · 10 performance vs area · 11 performance vs distance · 12 successful / unsuccessful cases.

Saved as PNG + PDF (+ CSV of the plotted data) in `figures/`.

In [ ]:
written = g01.figures()
show(['01_raw_signals', '02_magnetic_energy_scales', '03_gravity_energy_scales', '04_strike_coherence',
      '05_orientation_agreement', '06_single_vs_joint', '07_persistence', '08_six_way', '09_label_overlays',
      '10_performance_vs_area', '11_performance_vs_distance', '12_cases'])

## 6. Geological interpretation (write after reading the tables)

Answer from the verdict table and figures 9–12 — not from the exploratory maps:

* **A / B vs edge.** Is band-limited, multiscale energy better than a plain gradient? If not, the wavelet layers add cost, not information.
* **C vs A, B.** Does tolerant co-location help on *both* tasks? A gain only at d ≥ 500 m means the joint map is near faults but not on them.
* **D vs C.** Strike compatibility should remove co-located but discordant energy (e.g. a magnetic dyke crossing a gravity basin edge). No gain → the gate is too weak or orientation is uninformative at these scales.
* **E vs D.** Persistence should favour deep-rooted structures; a loss suggests real faults visible at one scale only (check the per-wavelength ablation).
* **E / C vs F.** Physical fusion vs M9's engineered features at equal area.
* **Failure cases (figure 12).** Contacts, intrusions, survey seams, data edges? These motivate G03 (fault vs contact).

In [ ]:
rep = g01.report()
display(Markdown(rep.read_text()[:8000]))
g01.push()
print('local :', g01.store.root)
print('Drive :', m9.cfg.get('_drive_root'), '/', g01.cfg['output_subdir'], '/', g01.store.root.name)

## Resume / reload without recomputation

After a disconnect: run the setup cells (0), then either re-run any section (finished steps are reused from the fingerprinted run folder) or attach to a finished run and read its results only:

In [ ]:
from gems_bench.m9.research.geology_io import list_runs, run_root
display(list_runs(run_root(m9, g01.cfg)))
g01_old = G01Experiment.attach(m9)                  # newest run; or attach(m9, run_id='g01_...')
old = g01_old.results()
display(old['hypothesis_contrasts'])
print(json.dumps(old['summary'], indent=1, default=str)[:3000])